# DirectDebit IQ — Failure Prediction Model

**Goal:** build an interview-ready machine learning workflow that predicts payment failure risk before collection.

This notebook covers:

1. Feature engineering  
2. Feature selection  
3. Model training and comparison  
4. XGBoost hyperparameter tuning  
5. Final model evaluation and export  

The modelling target is `is_failed`, where `1 = failed payment` and `0 = successful payment`.

## 0. Setup & Data Loading

We load the raw payment dataset, create a binary target, and prepare reusable project paths.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from src import config
from src.data_pipeline import clean_payments
from src.feature_store import FeatureStore

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

px.defaults.template = "plotly_white"
px.defaults.width = 950
px.defaults.height = 520

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "payments.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "failure_predictor.pkl"
METRICS_PATH = PROJECT_ROOT / "models" / "failure_predictor_metrics.json"
IMPORTANCE_PATH = PROJECT_ROOT / "models" / "failure_predictor_feature_importance.csv"

payments = pd.read_csv(DATA_PATH)
payments = clean_payments(payments)
payments["payment_date"] = pd.to_datetime(payments["payment_date"])
payments["is_failed"] = payments["payment_status"].eq("failed").astype(int)

print(f"Rows: {payments.shape[0]:,}")
print(f"Columns: {payments.shape[1]:,}")
display(payments.head())

## 1. Feature Engineering

We create seven predictive features requested for the project:

| Feature | Business meaning |
|---|---|
| `rolling_failure_rate` | Customer's failure rate across their previous 5 payments |
| `mandate_age_band` | Groups mandates into `new`, `young`, `mature`, and `established` |
| `is_monday` | Flags Monday collections because Monday has the highest failure pattern |
| `amount_zscore` | Shows how unusual a payment amount is for that customer |
| `days_since_failure` | Measures time since the customer's previous failed payment |
| `merchant_failure_rate` | Historical merchant failure rate before the current payment |
| `customer_lifetime_payments` | Total number of payments linked to the customer |

The rolling and merchant features are shifted so the current row does not leak the target into the model.

In [ ]:
store = FeatureStore()

# Build an interpretable feature frame for inspection.
feature_df = store._prepare_base_frame(payments)
feature_df = store.compute_rolling_failure_rate(feature_df, window=5)
feature_df = store.compute_merchant_risk_score(feature_df)
feature_df = store.compute_customer_lifetime_value(feature_df)
feature_df = store._add_mandate_age_band(feature_df)
feature_df = store._add_amount_zscore(feature_df)
feature_df = store._add_days_since_failure(feature_df)
feature_df = store._add_business_flags(feature_df)

engineered_features = [
    "customer_id",
    "merchant_id",
    "payment_date",
    "payment_amount",
    "payment_status",
    "rolling_failure_rate",
    "mandate_age_band",
    "is_monday",
    "amount_zscore",
    "days_since_failure",
    "merchant_failure_rate",
    "customer_lifetime_payments",
]

display(feature_df[engineered_features].head(10))

In [ ]:
# Quick quality check for engineered features
feature_quality = pd.DataFrame({
    "feature": engineered_features[5:],
    "missing_values": feature_df[engineered_features[5:]].isna().sum().values,
    "unique_values": feature_df[engineered_features[5:]].nunique().values,
})
display(feature_quality)

In [ ]:
# Visual check: average failure rate by mandate age band
mandate_chart = (
    feature_df.groupby("mandate_age_band", observed=False)["is_failed"]
    .mean()
    .mul(100)
    .reset_index(name="failure_rate_pct")
)
fig = px.bar(
    mandate_chart,
    x="mandate_age_band",
    y="failure_rate_pct",
    text="failure_rate_pct",
    title="Failure Rate by Mandate Age Band",
    labels={"mandate_age_band": "Mandate age band", "failure_rate_pct": "Failure rate (%)"},
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(yaxis_ticksuffix="%")
fig.show()

## 2. Feature Selection

We create the encoded model matrix, inspect correlation, estimate feature importance using a quick Random Forest, and remove highly correlated features.

In [ ]:
# Build model-ready matrix using the production FeatureStore class.
store = FeatureStore()
model_df = store.build_feature_matrix(payments)

X_full = model_df.drop(columns=["is_failed"])
y = model_df["is_failed"].astype(int)

print(f"Model matrix shape: {X_full.shape}")
print(f"Failure rate: {y.mean():.2%}")
display(X_full.head())

In [ ]:
# Correlation matrix on numeric features + target
corr_features = [
    "payment_amount",
    "log_payment_amount",
    "payment_day_of_month",
    "mandate_age_days",
    "previous_failure_count",
    "days_since_last_success",
    "rolling_failure_rate",
    "is_monday",
    "amount_zscore",
    "days_since_failure",
    "merchant_failure_rate",
    "customer_lifetime_payments",
    "customer_lifetime_value",
    "is_new_mandate",
    "has_multiple_previous_failures",
    "is_low_balance",
]

corr_df = model_df[[c for c in corr_features if c in model_df.columns] + ["is_failed"]].corr()
fig = px.imshow(
    corr_df,
    text_auto=".2f",
    aspect="auto",
    title="Correlation Matrix — Core Numeric Features",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
)
fig.update_layout(height=700)
fig.show()

In [ ]:
# Quick Random Forest importance to understand strongest predictors.
rf_quick = RandomForestClassifier(
    n_estimators=70,
    max_depth=7,
    min_samples_leaf=25,
    random_state=config.RANDOM_SEED,
    n_jobs=-1,
    class_weight="balanced_subsample",
)
rf_quick.fit(X_full, y)

importance_df = pd.DataFrame({
    "feature": X_full.columns,
    "importance": rf_quick.feature_importances_,
}).sort_values("importance", ascending=False)

display(importance_df.head(20))

fig = px.bar(
    importance_df.head(20).sort_values("importance"),
    x="importance",
    y="feature",
    orientation="h",
    title="Top 20 Features — Quick Random Forest Importance",
    labels={"importance": "Importance", "feature": "Feature"},
)
fig.show()

In [ ]:
# Remove highly correlated features using a conservative threshold.
CORRELATION_THRESHOLD = 0.92
absolute_corr = X_full.corr().abs()
upper_triangle = absolute_corr.where(np.triu(np.ones(absolute_corr.shape), k=1).astype(bool))

highly_correlated_features = [
    column for column in upper_triangle.columns if any(upper_triangle[column] > CORRELATION_THRESHOLD)
]

X = X_full.drop(columns=highly_correlated_features)
final_feature_list = list(X.columns)

print(f"Original features: {X_full.shape[1]}")
print(f"Removed highly correlated features: {len(highly_correlated_features)}")
print(f"Final features: {X.shape[1]}")
print("\nRemoved features:")
print(highly_correlated_features if highly_correlated_features else "None removed")

final_feature_table = pd.DataFrame({"final_feature": final_feature_list})
display(final_feature_table)

## 3. Model Training

We train and compare three models:

1. Logistic Regression  
2. Random Forest  
3. XGBoost  

Evaluation metrics:

- **AUC-ROC:** ranking quality across thresholds  
- **Average Precision:** better for imbalanced failure prediction  
- **F1 at threshold 0.3:** lower threshold catches more failures  
- **Revenue at risk caught per 1000 predictions:** business-facing metric

In [ ]:
THRESHOLD = 0.30
RECOVERY_COST_MULTIPLIER = 3.0

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=config.RANDOM_SEED,
    stratify=y,
)

# Needed for the business metric. This comes from X_test, so it is perfectly aligned.
amount_test = X_test["payment_amount"].copy()

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=config.RANDOM_SEED,
        )),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=120,
        max_depth=9,
        min_samples_leaf=20,
        random_state=config.RANDOM_SEED,
        n_jobs=-1,
        class_weight="balanced_subsample",
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.06,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=3,
        reg_lambda=1.0,
        eval_metric="logloss",
        random_state=config.RANDOM_SEED,
        n_jobs=2,
        tree_method="hist",
        verbosity=0,
        scale_pos_weight=scale_pos_weight,
    ),
}


def revenue_at_risk_caught_per_1000(y_true, probabilities, amounts, threshold=THRESHOLD):
    predicted_failure = probabilities >= threshold
    true_failure = y_true.to_numpy() == 1
    caught_value = amounts.to_numpy()[predicted_failure & true_failure].sum()
    return caught_value * RECOVERY_COST_MULTIPLIER / len(y_true) * 1000


results = []
fitted_models = {}
model_probabilities = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions_030 = (probabilities >= THRESHOLD).astype(int)

    fitted_models[model_name] = model
    model_probabilities[model_name] = probabilities

    results.append({
        "model": model_name,
        "AUC_ROC": roc_auc_score(y_test, probabilities),
        "Average_Precision": average_precision_score(y_test, probabilities),
        "F1_at_0_30": f1_score(y_test, predictions_030, zero_division=0),
        "Precision_at_0_30": precision_score(y_test, predictions_030, zero_division=0),
        "Recall_at_0_30": recall_score(y_test, predictions_030, zero_division=0),
        "Revenue_at_Risk_Caught_per_1000": revenue_at_risk_caught_per_1000(
            y_test, probabilities, amount_test, THRESHOLD
        ),
    })

results_df = pd.DataFrame(results).sort_values("Average_Precision", ascending=False)
display(results_df)

In [ ]:
fig = px.bar(
    results_df.melt(
        id_vars="model",
        value_vars=["AUC_ROC", "Average_Precision", "F1_at_0_30", "Recall_at_0_30"],
        var_name="metric",
        value_name="score",
    ),
    x="model",
    y="score",
    color="metric",
    barmode="group",
    title="Model Comparison — Predictive Metrics",
    labels={"model": "Model", "score": "Score", "metric": "Metric"},
)
fig.update_layout(yaxis_range=[0, 1])
fig.show()

In [ ]:
fig = px.bar(
    results_df.sort_values("Revenue_at_Risk_Caught_per_1000"),
    x="Revenue_at_Risk_Caught_per_1000",
    y="model",
    orientation="h",
    title="Business Metric — Revenue at Risk Caught per 1,000 Predictions",
    labels={
        "Revenue_at_Risk_Caught_per_1000": "Estimated value caught per 1,000 predictions",
        "model": "Model",
    },
)
fig.show()

In [ ]:
# ROC curves
fig = go.Figure()
for model_name, probabilities in model_probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=model_name))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random baseline", line=dict(dash="dash")))
fig.update_layout(
    title="ROC Curves — Failure Prediction Models",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=550,
)
fig.show()

## 4. Hyperparameter Tuning on XGBoost

XGBoost is tuned using a compact randomized search. This keeps the notebook practical while still showing a professional model optimisation workflow.

In [ ]:
# Use a capped sample for fast, repeatable tuning in a portfolio notebook.
TUNING_SAMPLE_SIZE = min(10_000, len(X_train))
X_tune = X_train.sample(TUNING_SAMPLE_SIZE, random_state=config.RANDOM_SEED)
y_tune = y_train.loc[X_tune.index]

xgb_base = XGBClassifier(
    eval_metric="logloss",
    random_state=config.RANDOM_SEED,
    n_jobs=2,
    tree_method="hist",
    verbosity=0,
    scale_pos_weight=scale_pos_weight,
)

param_distributions = {
    "n_estimators": [80, 120, 160],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.05, 0.08],
    "subsample": [0.75, 0.85, 0.95],
    "colsample_bytree": [0.75, 0.85, 0.95],
    "min_child_weight": [1, 3, 5],
    "reg_lambda": [0.5, 1.0, 2.0],
}

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_distributions,
    n_iter=3,
    scoring="average_precision",
    cv=2,
    random_state=config.RANDOM_SEED,
    verbose=0,
    n_jobs=1,
)

search.fit(X_tune, y_tune)

print("Best average precision CV score:", round(search.best_score_, 4))
print("Best parameters:")
search.best_params_

In [ ]:
tuned_xgb = search.best_estimator_
tuned_xgb.fit(X_train, y_train)

tuned_probabilities = tuned_xgb.predict_proba(X_test)[:, 1]
tuned_predictions_030 = (tuned_probabilities >= THRESHOLD).astype(int)

tuned_metrics = {
    "model": "Tuned XGBoost",
    "AUC_ROC": roc_auc_score(y_test, tuned_probabilities),
    "Average_Precision": average_precision_score(y_test, tuned_probabilities),
    "F1_at_0_30": f1_score(y_test, tuned_predictions_030, zero_division=0),
    "Precision_at_0_30": precision_score(y_test, tuned_predictions_030, zero_division=0),
    "Recall_at_0_30": recall_score(y_test, tuned_predictions_030, zero_division=0),
    "Revenue_at_Risk_Caught_per_1000": revenue_at_risk_caught_per_1000(
        y_test, tuned_probabilities, amount_test, THRESHOLD
    ),
}

comparison_with_tuned = pd.concat(
    [results_df, pd.DataFrame([tuned_metrics])],
    ignore_index=True,
).sort_values("Average_Precision", ascending=False)

display(comparison_with_tuned)

## 5. Final Model Evaluation

We evaluate the tuned XGBoost model at threshold `0.30`, which intentionally catches more likely failures for proactive payment recovery workflows.

In [ ]:
cm = confusion_matrix(y_test, tuned_predictions_030)
cm_df = pd.DataFrame(
    cm,
    index=["Actual success", "Actual failed"],
    columns=["Predicted success", "Predicted failed"],
)

fig = px.imshow(
    cm_df,
    text_auto=True,
    title="Confusion Matrix — Tuned XGBoost at Threshold 0.30",
    color_continuous_scale="Blues",
)
fig.show()

display(pd.DataFrame(classification_report(y_test, tuned_predictions_030, output_dict=True, zero_division=0)).T)

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, tuned_probabilities)

fig = go.Figure()
fig.add_trace(go.Scatter(x=recall, y=precision, mode="lines", name="Precision-Recall curve"))
fig.update_layout(
    title="Precision-Recall Curve — Tuned XGBoost",
    xaxis_title="Recall",
    yaxis_title="Precision",
    height=520,
)
fig.show()

In [ ]:
# Final feature importance
final_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": tuned_xgb.feature_importances_,
}).sort_values("importance", ascending=False)

display(final_importance.head(25))

fig = px.bar(
    final_importance.head(25).sort_values("importance"),
    x="importance",
    y="feature",
    orientation="h",
    title="Final Tuned XGBoost — Top 25 Feature Importances",
)
fig.show()

In [ ]:
# Save final model artifact
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

artifact = {
    "model": tuned_xgb,
    "feature_columns": list(X.columns),
    "removed_highly_correlated_features": highly_correlated_features,
    "threshold": THRESHOLD,
    "target_column": "is_failed",
    "model_type": "Tuned XGBClassifier",
    "best_params": search.best_params_,
    "metrics": tuned_metrics,
}

joblib.dump(artifact, MODEL_PATH)

pd.Series(tuned_metrics).to_json(METRICS_PATH, indent=2)
final_importance.to_csv(IMPORTANCE_PATH, index=False)

print(f"Saved final model to: {MODEL_PATH}")
print(f"Saved metrics to: {METRICS_PATH}")
print(f"Saved feature importance to: {IMPORTANCE_PATH}")

## Final Business Takeaways

- **Customer history is one of the strongest signals:** customers with recent or repeated failures should be monitored before collection attempts.
- **Merchant-level patterns matter:** some merchants consistently carry higher failure risk, which can guide account management and operational reviews.
- **New mandates are riskier:** early-stage mandates should receive stronger validation, reminders, or pre-checks before payment collection.
- **A lower threshold is useful for operations:** threshold `0.30` catches more likely failures, even if it creates more false positives.
- **Average Precision is more useful than accuracy here:** payment failures are the minority class, so accuracy alone can look good while missing risk.
- **The business metric converts ML into money:** revenue-at-risk caught per 1,000 predictions helps stakeholders understand model value.